# Hate Speech Classification - LLM Fine-Tuning with LoRA (M3 Optimized)

This notebook fine-tunes small language models (Qwen3, Phi, Llama, etc.) for hate speech classification using **LoRA** on Apple Silicon.

## Prerequisites & Setup

1. **Python Environment**:
    ```bash
    python3 -m venv .venv-llm-ft
    source .venv-llm-ft/bin/activate
    ```

2. **Install Dependencies**:
    ```bash
    pip install mlx mlx-lm transformers datasets scikit-learn pandas tqdm ipykernel
    ```

3. **Register Kernel**:
    ```bash
    python -m ipykernel install --user --name=.venv-llm-ft
    ```

## Supported Models (MLX-Community 4-bit versions)

| Model | MLX HuggingFace ID | Size |
|-------|-------------------|------|
| Phi-3 Mini | `mlx-community/Phi-3-mini-4k-instruct-4bit` | 3.8B |
| Qwen2.5 | `mlx-community/Qwen2.5-3B-Instruct-4bit` | 3B |
| Gemma 2 | `mlx-community/gemma-2-2b-it-4bit` | 2B |

In [ ]:
'''
python -c "
from huggingface_hub import snapshot_download
import sys
print('📥 Starting download of Phi-3-mini-4k-instruct-4bit...', flush=True)
print('This will take a few minutes (~2-3 GB)...', flush=True)
path = snapshot_download('mlx-community/Qwen2.5-3B-Instruct-4bit')
print(f'✅ Downloaded to: {path}', flush=True)
"
'''


In [ ]:
import os
import json
import pandas as pd
from datasets import load_dataset
from datetime import datetime

# --- Quick Config --- #

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit" 
DATA_DIR = "data"
ADAPTER_PATH = f"./adapters/{MODEL_NAME.split('/')[-1]}-hatexplain"
NUM_ITERS = 500
BATCH_SIZE = 4
LORA_LAYERS = 8
print(f"✅ Config ready | Model: {MODEL_NAME.split('/')[-1]}")

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-llm-ft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Config ready | Model: Qwen2.5-3B-Instruct-4bit


## 1. Prepare Training Data

MLX-LM expects data in a specific chat format. We'll convert our HateXplain dataset.

In [3]:
# Label mappings
ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}

def get_label(annotators):
    """Majority vote from annotators."""
    if not annotators:
        return 1
    labels = annotators.get("label", []) if isinstance(annotators, dict) else []
    if not labels:
        return 1
    counts = {0: 0, 1: 0, 2: 0}
    for lab in labels:
        lab = int(lab)
        if lab in counts:
            counts[lab] += 1
    return max(counts, key=lambda k: (counts[k], 2-k))


def format_example(example):
    """Format as chat messages for MLX-LM."""
    text = " ".join(example["post_tokens"]) if isinstance(example.get("post_tokens"), list) else ""
    label_id = get_label(example.get("annotators", {}))
    
    # Skip offensive for binary classification
    if label_id == 2:
        return None
    
    label_name = ID2NAME[label_id]
    
    return {
        "messages": [
            {"role": "system", "content": "Classify as 'hatespeech' or 'normal'. Reply with only the label."},
            {"role": "user", "content": text},
            {"role": "assistant", "content": label_name}
        ]
    }

In [4]:
# Load dataset
print("Loading HateXplain dataset...")

data_files = {
    "train": "hatexplain/hatexplain_train.jsonl",
    "validation": "hatexplain/hatexplain_validation.jsonl",
    "test": "hatexplain/hatexplain_test.jsonl"
}

if all(os.path.exists(p) for p in data_files.values()):
    print("✅ Loading from local JSONL files...")
    ds = load_dataset("json", data_files=data_files)
else:
    print("⚠️ Local files not found. Loading from HuggingFace...")
    ds = load_dataset("hatexplain")

print(f"Loaded: {len(ds['train'])} train, {len(ds['validation'])} val, {len(ds['test'])} test")

Loading HateXplain dataset...
✅ Loading from local JSONL files...
Loaded: 15383 train, 1922 val, 1924 test


In [5]:
# Convert and save as JSONL
os.makedirs(DATA_DIR, exist_ok=True)

def save_split(dataset, filename):
    path = os.path.join(DATA_DIR, filename)
    count = 0
    with open(path, 'w') as f:
        for ex in dataset:
            formatted = format_example(ex)
            if formatted:
                f.write(json.dumps(formatted) + '\n')
                count += 1
    print(f"✅ Saved {count} examples to {path}")

save_split(ds["train"], "train.jsonl")
save_split(ds["validation"], "valid.jsonl")
save_split(ds["test"], "test.jsonl")

✅ Saved 10999 examples to data/train.jsonl
✅ Saved 1374 examples to data/valid.jsonl
✅ Saved 1376 examples to data/test.jsonl


## 2. Fine-Tune with MLX-LM

Run LoRA fine-tuning using MLX-LM's command line interface.

In [7]:
# Check MLX installation
try:
    import mlx.core as mx
    import mlx_lm
    print(f"✅ MLX installed (Apple Silicon ready)")
    print(f"✅ MLX-LM ready")
except ImportError as e:
    print(f"❌ Missing: {e}")
    print("Install with: pip install mlx mlx-lm")

✅ MLX installed (Apple Silicon ready)
✅ MLX-LM ready


In [8]:
# Generate training command
train_cmd = f"""python -m mlx_lm.lora \\
    --model {MODEL_NAME} \\
    --train \\
    --data {DATA_DIR} \\
    --batch-size {BATCH_SIZE} \\
    --lora-layers {LORA_LAYERS} \\
    --iters {NUM_ITERS} \\
    --adapter-path {ADAPTER_PATH}"""

print("🚀 Run this command in terminal to start training:")
print("\n" + "="*60)
print(train_cmd)
print("="*60)
print("\nEstimated time: 10-30 minutes on M3")

🚀 Run this command in terminal to start training:

python -m mlx_lm.lora \
    --model mlx-community/Qwen2.5-3B-Instruct-4bit \
    --train \
    --data data \
    --batch-size 4 \
    --lora-layers 8 \
    --iters 500 \
    --adapter-path ./adapters/Qwen2.5-3B-Instruct-4bit-hatexplain

Estimated time: 10-30 minutes on M3


In [10]:
# Alternative: Run training directly from notebook (may take a while)
# Uncomment to run:

import subprocess
result = subprocess.run(
    ["mlx_lm.lora",
     "--model", MODEL_NAME,
     "--train",
     "--data", DATA_DIR,
     "--batch-size", str(BATCH_SIZE),
     "--num-layers", str(LORA_LAYERS),
     "--iters", str(NUM_ITERS),
     "--adapter-path", ADAPTER_PATH],
    capture_output=False
)

# print("💡 Uncomment the code above to run training from notebook")

Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 38756.40it/s]


Loading datasets
Training
Trainable parameters: 0.108% (3.326M/3085.939M)
Starting training..., iters: 500


mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.
Calculating loss...: 100%|██████████| 25/25 [00:22<00:00,  1.12it/s]


Iter 1: Val loss 6.353, Val took 22.345s
Iter 10: Train loss 4.897, Learning Rate 1.000e-05, It/sec 0.630, Tokens/sec 170.252, Trained Tokens 2703, Peak mem 3.375 GB
Iter 20: Train loss 2.835, Learning Rate 1.000e-05, It/sec 0.751, Tokens/sec 187.078, Trained Tokens 5193, Peak mem 3.375 GB
Iter 30: Train loss 2.146, Learning Rate 1.000e-05, It/sec 0.732, Tokens/sec 165.312, Trained Tokens 7450, Peak mem 3.375 GB
Iter 40: Train loss 2.024, Learning Rate 1.000e-05, It/sec 0.723, Tokens/sec 179.919, Trained Tokens 9940, Peak mem 3.375 GB
Iter 50: Train loss 2.046, Learning Rate 1.000e-05, It/sec 0.702, Tokens/sec 173.779, Trained Tokens 12417, Peak mem 3.375 GB
Iter 60: Train loss 1.902, Learning Rate 1.000e-05, It/sec 0.706, Tokens/sec 164.895, Trained Tokens 14754, Peak mem 3.375 GB
Iter 70: Train loss 2.147, Learning Rate 1.000e-05, It/sec 0.596, Tokens/sec 152.914, Trained Tokens 17320, Peak mem 3.376 GB
Iter 80: Train loss 2.290, Learning Rate 1.000e-05, It/sec 0.636, Tokens/sec 164.

Calculating loss...: 100%|██████████| 25/25 [00:32<00:00,  1.28s/it]


Iter 200: Val loss 2.025, Val took 32.074s
Iter 200: Train loss 1.867, Learning Rate 1.000e-05, It/sec 0.546, Tokens/sec 132.752, Trained Tokens 50870, Peak mem 3.703 GB
Iter 200: Saved adapter weights to adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/adapters.safetensors and adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/0000200_adapters.safetensors.
Iter 210: Train loss 2.078, Learning Rate 1.000e-05, It/sec 0.272, Tokens/sec 79.246, Trained Tokens 53781, Peak mem 6.562 GB
Iter 220: Train loss 1.981, Learning Rate 1.000e-05, It/sec 0.415, Tokens/sec 104.690, Trained Tokens 56303, Peak mem 6.562 GB
Iter 230: Train loss 2.062, Learning Rate 1.000e-05, It/sec 0.455, Tokens/sec 115.724, Trained Tokens 58844, Peak mem 6.562 GB
Iter 240: Train loss 2.041, Learning Rate 1.000e-05, It/sec 0.490, Tokens/sec 123.879, Trained Tokens 61372, Peak mem 6.562 GB
Iter 250: Train loss 2.024, Learning Rate 1.000e-05, It/sec 0.410, Tokens/sec 105.022, Trained Tokens 63936, Peak mem 6.562 GB
Iter 260: Train 

Calculating loss...: 100%|██████████| 25/25 [00:37<00:00,  1.48s/it]


Iter 400: Val loss 2.080, Val took 37.130s
Iter 400: Train loss 1.917, Learning Rate 1.000e-05, It/sec 0.498, Tokens/sec 119.381, Trained Tokens 101807, Peak mem 6.562 GB
Iter 400: Saved adapter weights to adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/adapters.safetensors and adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/0000400_adapters.safetensors.
Iter 410: Train loss 1.903, Learning Rate 1.000e-05, It/sec 0.456, Tokens/sec 113.301, Trained Tokens 104290, Peak mem 6.562 GB
Iter 420: Train loss 2.185, Learning Rate 1.000e-05, It/sec 0.362, Tokens/sec 94.292, Trained Tokens 106892, Peak mem 6.562 GB
Iter 430: Train loss 2.075, Learning Rate 1.000e-05, It/sec 0.356, Tokens/sec 96.204, Trained Tokens 109593, Peak mem 6.562 GB
Iter 440: Train loss 1.855, Learning Rate 1.000e-05, It/sec 0.475, Tokens/sec 117.433, Trained Tokens 112066, Peak mem 6.562 GB
Iter 450: Train loss 2.010, Learning Rate 1.000e-05, It/sec 0.471, Tokens/sec 127.556, Trained Tokens 114772, Peak mem 6.562 GB
Iter 460: T

Calculating loss...: 100%|██████████| 25/25 [00:34<00:00,  1.38s/it]


Iter 500: Val loss 1.905, Val took 34.420s
Iter 500: Train loss 1.983, Learning Rate 1.000e-05, It/sec 0.494, Tokens/sec 126.476, Trained Tokens 127519, Peak mem 6.562 GB
Iter 500: Saved adapter weights to adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/adapters.safetensors and adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/0000500_adapters.safetensors.
Saved final weights to adapters/Qwen2.5-3B-Instruct-4bit-hatexplain/adapters.safetensors.


## 3. Test Fine-Tuned Model

In [11]:
from mlx_lm import load, generate

# Load model with adapter
print(f"Loading {MODEL_NAME} with adapter...")

try:
    model, tokenizer = load(MODEL_NAME, adapter_path=ADAPTER_PATH)
    print("✅ Model loaded with LoRA adapter")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure training has completed first.")

Loading mlx-community/Qwen2.5-3B-Instruct-4bit with adapter...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 160632.92it/s]


✅ Model loaded with LoRA adapter


In [12]:
def classify(text):
    """Classify text using fine-tuned model."""
    messages = [
        {"role": "system", "content": "Classify as 'hatespeech' or 'normal'. Reply with only the label."},
        {"role": "user", "content": text}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = generate(model, tokenizer, prompt=prompt, max_tokens=10)
    
    # Parse response
    pred = response.strip().lower()
    if "hatespeech" in pred:
        return "hatespeech"
    elif "normal" in pred:
        return "normal"
    return pred

# Test examples
test_texts = [
    "I love everyone regardless of their background.",
    "Those people should all go back where they came from.",
    "Great weather today!"
]

print("\n" + "="*50)
print("EXAMPLE PREDICTIONS")
print("="*50)
for text in test_texts:
    pred = classify(text)
    print(f"\nText: {text}")
    print(f"  → {pred}")


EXAMPLE PREDICTIONS


mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.



Text: I love everyone regardless of their background.
  → normal

Text: Those people should all go back where they came from.
  → hatespeech

Text: Great weather today!
  → normal


## 4. Evaluate on Test Set

In [13]:
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import csv

# Load test data
test_examples = []
with open(os.path.join(DATA_DIR, "test.jsonl"), 'r') as f:
    for line in f:
        ex = json.loads(line)
        test_examples.append({
            "text": ex["messages"][1]["content"],
            "label": ex["messages"][2]["content"]
        })

print(f"Loaded {len(test_examples)} test examples")

# Limit for faster evaluation (remove slice for full eval)
EVAL_LIMIT = 200
test_subset = test_examples[:EVAL_LIMIT]
print(f"Evaluating on {len(test_subset)} examples...")

Loaded 1376 test examples
Evaluating on 200 examples...


In [14]:
# Run evaluation
y_true = []
y_pred = []

for ex in tqdm(test_subset):
    pred = classify(ex["text"])
    y_true.append(ex["label"])
    y_pred.append(pred)

print("\n✅ Evaluation complete")

100%|██████████| 200/200 [01:02<00:00,  3.18it/s]


✅ Evaluation complete


In [15]:
# Pretty print results
report_dict = classification_report(y_true, y_pred, target_names=["hatespeech", "normal"], output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=["hatespeech", "normal"])

print("\n" + "═" * 60)
print("  📊 LLM FINE-TUNE RESULTS")
print("═" * 60)
print(f"\n  Model: {MODEL_NAME}")
print(f"  Samples: {len(y_true)}")

print("\n┌" + "─" * 58 + "┐")
print("│" + " OVERALL METRICS".center(58) + "│")
print("├" + "─" * 29 + "┬" + "─" * 28 + "┤")
print(f"│  Accuracy:  {report_dict['accuracy']*100:>6.2f}%" + " " * 12 + "│" + f"  Macro F1:  {report_dict['macro avg']['f1-score']*100:>6.2f}%" + " " * 6 + "│")
print("└" + "─" * 29 + "┴" + "─" * 28 + "┘")

print("\n┌" + "─" * 58 + "┐")
print("│" + " PER-CLASS METRICS".center(58) + "│")
print("├" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 13 + "┤")
print("│" + "  Class".center(14) + "│" + "  Precision".center(14) + "│" + "  Recall".center(14) + "│" + "  F1-Score".center(13) + "│")
print("├" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 13 + "┤")
for class_name in ["hatespeech", "normal"]:
    m = report_dict[class_name]
    print(f"│  {class_name:<11} │  {m['precision']*100:>6.2f}%    │  {m['recall']*100:>6.2f}%    │  {m['f1-score']*100:>5.2f}%   │")
print("└" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 13 + "┘")

print("\n┌" + "─" * 38 + "┐")
print("│" + " CONFUSION MATRIX".center(38) + "│")
print("├" + "─" * 38 + "┤")
print("│              Predicted".ljust(38) + "│")
print("│           hatespeech   normal".ljust(38) + "│")
print("├" + "─" * 38 + "┤")
print(f"│  Actual hatespeech │  {cm[0][0]:>5}   │  {cm[0][1]:>5}   │")
print(f"│         normal     │  {cm[1][0]:>5}   │  {cm[1][1]:>5}   │")
print("└" + "─" * 38 + "┘")


════════════════════════════════════════════════════════════
  📊 LLM FINE-TUNE RESULTS
════════════════════════════════════════════════════════════

  Model: mlx-community/Qwen2.5-3B-Instruct-4bit
  Samples: 200

┌──────────────────────────────────────────────────────────┐
│                      OVERALL METRICS                     │
├─────────────────────────────┬────────────────────────────┤
│  Accuracy:   76.50%            │  Macro F1:   74.87%      │
└─────────────────────────────┴────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│                     PER-CLASS METRICS                    │
├──────────────┬──────────────┬──────────────┬─────────────┤
│     Class    │   Precision  │     Recall   │    F1-Score │
├──────────────┼──────────────┼──────────────┼─────────────┤
│  hatespeech  │   83.61%    │   57.95%    │  68.46%   │
│  normal      │   73.38%    │   91.07%    │  81.27%   │
└──────────────┴──────────────┴──────────────┴─────────────┘

In [ ]:
# Save results to CSV
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(results_dir, "model_scores.csv")

result_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_name": MODEL_NAME,
    "model_type": "fine-tuned (llm)", 
    "task": "binary",
    "epochs": "N/A",
    "batch_size": BATCH_SIZE,
    "learning_rate": "N/A",
    "max_length": "N/A",
    "accuracy": round(report_dict["accuracy"], 4),
    "macro_f1": round(report_dict["macro avg"]["f1-score"], 4),
    "macro_precision": round(report_dict["macro avg"]["precision"], 4),
    "macro_recall": round(report_dict["macro avg"]["recall"], 4),
    "hatespeech_f1": round(report_dict["hatespeech"]["f1-score"], 4),
    "hatespeech_precision": round(report_dict["hatespeech"]["precision"], 4),
    "hatespeech_recall": round(report_dict["hatespeech"]["recall"], 4),
    "normal_f1": round(report_dict["normal"]["f1-score"], 4),
    "normal_precision": round(report_dict["normal"]["precision"], 4),
    "normal_recall": round(report_dict["normal"]["recall"], 4),
    "cm_hate_as_hate": int(cm[0][0]),
    "cm_hate_as_normal": int(cm[0][1]),
    "cm_normal_as_hate": int(cm[1][0]),
    "cm_normal_as_normal": int(cm[1][1]),
    "test_samples": len(y_true),
}

file_exists = os.path.exists(results_file)
with open(results_file, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=result_entry.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(result_entry)

print("\n" + "═" * 60)
print(f"  ✅ Results saved to: {results_file}")
print("═" * 60)


════════════════════════════════════════════════════════════
  ✅ Results saved to: results/model_scores.csv
════════════════════════════════════════════════════════════
